# Trust Region Q Adjoint Matching Executable Report

This notebook is the single control surface for the TRQAM full-run workflow. Running all cells can generate the Slurm job scripts, submit the full BC and TRQAM jobs, and later render the evaluation CSVs into a report.

The default profile is `full_pipeline`: the notebook submits BC pretraining first, then submits TRQAM fine-tuning with an `afterok` dependency on the BC job. Training itself runs inside Slurm jobs, so the notebook kernel does not need to stay alive for the whole multi-day run.


In [ ]:
# Report display controls. Set REPORT_MODE=False before Run All if you want code cells visible.
REPORT_MODE = True

from IPython.display import HTML, display

REPORT_CSS = r'''
<style>
:root {
  --trqam-text: #1f2328;
  --trqam-muted: #57606a;
  --trqam-border: #d0d7de;
  --trqam-accent: #0969da;
  --trqam-bg: #ffffff;
  --trqam-soft: #f6f8fa;
}
body, .jp-Notebook, .notebook_app {
  color: var(--trqam-text);
}
.trqam-callout {
  border-left: 4px solid var(--trqam-accent);
  background: var(--trqam-soft);
  padding: 12px 14px;
  margin: 10px 0 16px;
}
.trqam-kpi-grid {
  display: grid;
  grid-template-columns: repeat(auto-fit, minmax(150px, 1fr));
  gap: 10px;
  margin: 12px 0 18px;
}
.trqam-kpi {
  border: 1px solid var(--trqam-border);
  border-radius: 6px;
  padding: 10px 12px;
  background: var(--trqam-bg);
}
.trqam-kpi .label {
  color: var(--trqam-muted);
  font-size: 12px;
  margin-bottom: 4px;
}
.trqam-kpi .value {
  font-weight: 650;
  font-size: 18px;
}
.trqam-report h2, .trqam-report h3 {
  margin-top: 18px;
}
.trqam-report table, table.dataframe, .report-table {
  border-collapse: collapse;
  width: 100%;
  font-size: 13px;
}
.trqam-report th, .trqam-report td, table.dataframe th, table.dataframe td, .report-table th, .report-table td {
  border: 1px solid var(--trqam-border);
  padding: 6px 8px;
  text-align: left;
  vertical-align: top;
}
.trqam-report th, table.dataframe th, .report-table th {
  background: var(--trqam-soft);
}
.trqam-muted { color: var(--trqam-muted); }
@media print {
  .jp-Toolbar, .jp-NotebookPanel-toolbar, .jp-Cell-inputWrapper, .jp-InputPrompt, .jp-OutputPrompt, div.input, .prompt {
    display: none !important;
  }
}
</style>
'''

HIDE_CODE_CSS = r'''
<style id="trqam-hide-code">
.jp-Cell-inputWrapper, .jp-InputPrompt, .jp-OutputPrompt, div.input, .prompt {
  display: none !important;
}
</style>
'''

if REPORT_MODE:
    display(HTML(REPORT_CSS + HIDE_CODE_CSS))
else:
    display(HTML(REPORT_CSS))


## 1. Background

TRQAM fine-tunes a pretrained flow policy with off-policy RL while constraining the updated policy to stay close to the base policy under a path-space KL trust region. The Q-function action gradient is used as an adjoint signal for improving the policy vector field. When the estimated KL exceeds the configured budget, dual descent increases the trust-region coefficient and makes the policy update more conservative.

The paper pipeline has two stages.

1. Train a flow policy with behavior cloning by setting `bc_only=True`.
2. Load the BC checkpoint as `pretrained_actor_path` and fine-tune with TRQAM.


In [ ]:
from pathlib import Path
from IPython.display import Image, Markdown, display

main_figure = Path('assets/Main_figure.png')
if main_figure.exists():
    display(Image(filename=str(main_figure)))
else:
    display(Markdown('_Main figure image was not found at `assets/Main_figure.png`._'))


## 2. Runtime Assumptions

This notebook assumes it is executed from the repository root with the `Python (trqam)` Jupyter kernel. The OGBench dependencies are installed in the `trqam` conda environment. RoboMimic environments such as `lift-mh-low_dim` require the additional robomimic/robosuite setup described in the README.

For GPU execution, the Jupyter server or kernel must run inside a GPU allocation. If the notebook is opened on a login node, JAX may only see the CPU. The notebook sets `MUJOCO_GL=egl` for headless MuJoCo rendering and `XLA_PYTHON_CLIENT_PREALLOCATE=false` to avoid aggressive JAX GPU memory preallocation.


In [ ]:
import importlib.util
import os

os.environ.setdefault('WANDB_MODE', 'disabled')
os.environ.setdefault('MUJOCO_GL', 'egl')
os.environ.setdefault('XLA_PYTHON_CLIENT_PREALLOCATE', 'false')

required_packages = ['jax', 'flax', 'distrax', 'pandas', 'matplotlib', 'tqdm', 'ml_collections', 'ogbench']
missing_packages = [pkg for pkg in required_packages if importlib.util.find_spec(pkg) is None]
if missing_packages:
    raise RuntimeError(
        'Missing packages: ' + ', '.join(missing_packages) + '\n'
        'Activate the TRQAM environment or install dependencies with: pip install -r requirements.txt'
    )

import glob
import html
import json
import pickle
import random
import re
import sys
import time
from collections import defaultdict
from dataclasses import asdict, dataclass
from pathlib import Path
from typing import Optional

import flax
import jax
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import HTML, Markdown, display
from tqdm.auto import trange

ROOT = Path.cwd()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from agents import agents
from agents.trqam import get_config as get_trqam_config
from envs.env_utils import make_env_and_datasets
from envs.ogbench_utils import make_ogbench_env_and_datasets
from evaluation import evaluate
from utils.datasets import Dataset, ReplayBuffer
from utils.flax_utils import save_agent


def is_robomimic_env(env_name):
    if 'low_dim' not in env_name:
        return False
    task, dataset_type, _ = env_name.split('-')
    return task in ('lift', 'can', 'square', 'transport', 'tool_hang') and dataset_type in ('mh', 'ph')


def display_kv(title, rows):
    df = pd.DataFrame(rows, columns=['Item', 'Value'])
    display(Markdown(f'### {title}'))
    display(df)
    return df


def fmt_value(value):
    if isinstance(value, float):
        return f'{value:.6g}'
    if isinstance(value, (tuple, list)):
        return ', '.join(map(str, value))
    return str(value)

runtime_info = {
    'workspace': str(ROOT),
    'jax_backend': jax.default_backend(),
    'jax_devices': ', '.join(str(device) for device in jax.devices()),
    'wandb_mode': os.environ.get('WANDB_MODE', ''),
    'mujoco_gl': os.environ.get('MUJOCO_GL', ''),
}

display_kv(
    'Runtime Check',
    [
        ('Workspace', f'`{runtime_info["workspace"]}`'),
        ('JAX backend', f'`{runtime_info["jax_backend"]}`'),
        ('JAX devices', f'`{runtime_info["jax_devices"]}`'),
        ('WANDB_MODE', f'`{runtime_info["wandb_mode"]}`'),
        ('MUJOCO_GL', f'`{runtime_info["mujoco_gl"]}`'),
    ],
)

if runtime_info['jax_backend'] != 'gpu':
    display(Markdown('> **Warning:** this notebook kernel does not currently see a JAX GPU backend. Start the Jupyter kernel inside a GPU allocation for GPU execution.'))


## 3. Experiment Settings

`NotebookConfig` mirrors the main `main.py` flags in a form that is easy to edit inside the notebook. The default mode is a Slurm-controlled full pipeline, not direct in-kernel training.

- `RUN_PROFILE='full_pipeline'`: submit BC pretraining and dependent TRQAM fine-tuning from this notebook
- `RUN_PROFILE='full_bc'`: submit only the 300K-step BC pretraining job
- `RUN_PROFILE='full_trqam'`: submit only the 1M offline + 500K online TRQAM job from an existing BC checkpoint
- `RUN_PROFILE='smoke'`: run a short notebook-local smoke test

Set `SUBMIT_SLURM_JOB=False` if you want Run All to only generate scripts without submitting. Set `EXECUTION_MODE='notebook'` only for short interactive runs; full runs should use `EXECUTION_MODE='slurm'`.


In [ ]:
@dataclass
class NotebookConfig:
    run_group: str = 'notebook_report'
    tags: str = 'TRQAM_NOTEBOOK'
    seed: int = 10001
    env_name: str = 'cube-triple-play-singletask-task2-v0'
    save_dir: str = 'exp_full'
    ogbench_dataset_dir: Optional[str] = None
    pretrained_actor_path: Optional[str] = None

    offline_steps: int = 1
    online_steps: int = 0
    log_interval: int = 1
    eval_interval: int = 1
    save_interval: int = 0
    eval_episodes: int = 1
    video_episodes: int = 0
    video_frame_skip: int = 3

    horizon_length: int = 5
    sparse: bool = False
    action_chunking: bool = True
    bc_only: bool = True
    kl_budget: float = 0.5
    kl_budget_online: Optional[float] = None

    dataset_proportion: float = 0.001
    dataset_replace_interval: int = 0
    buffer_size: int = 1_000_000
    start_training: int = 5
    utd_ratio: int = 1
    balanced_sampling: bool = False
    large_network: bool = False

    batch_size: int = 8
    actor_hidden_dims: tuple = (64, 64)
    value_hidden_dims: tuple = (64, 64)
    actor_layer_norm: bool = False
    flow_steps: int = 2


# Main notebook controls.
RUN_PROFILE = os.environ.get('TRQAM_NOTEBOOK_PROFILE', 'full_pipeline')
EXECUTION_MODE = os.environ.get(
    'TRQAM_EXECUTION_MODE',
    'notebook' if RUN_PROFILE == 'smoke' else 'slurm',
)
SUBMIT_SLURM_JOB = os.environ.get('TRQAM_SUBMIT_SLURM', 'true').lower() in {'1', 'true', 'yes'}
WAIT_FOR_SLURM_COMPLETION = os.environ.get('TRQAM_WAIT_FOR_SLURM', 'false').lower() in {'1', 'true', 'yes'}

PRETRAINED_ACTOR_PATH = os.environ.get('TRQAM_PRETRAINED_ACTOR_PATH') or None
OGBENCH_DATASET_DIR = os.environ.get('TRQAM_OGBENCH_DATASET_DIR') or None

SLURM_PARTITION = os.environ.get('TRQAM_SLURM_PARTITION', '4090')
SLURM_GPUS = os.environ.get('TRQAM_SLURM_GPUS', 'gpu:1')
SLURM_CPUS = int(os.environ.get('TRQAM_SLURM_CPUS', '8'))
SLURM_MEM = os.environ.get('TRQAM_SLURM_MEM', '64G')
SLURM_TIME = os.environ.get('TRQAM_SLURM_TIME', '3-00:00:00')

FULL_WIDTH = int(os.environ.get('TRQAM_FULL_WIDTH', '512'))
FULL_HIDDEN_DIMS = (FULL_WIDTH, FULL_WIDTH, FULL_WIDTH, FULL_WIDTH)
FULL_ACTOR_LAYER_NORM = os.environ.get('TRQAM_ACTOR_LAYER_NORM', 'False').lower() in {'1', 'true', 'yes'}
FULL_DATASET_REPLACE_INTERVAL = int(os.environ.get('TRQAM_DATASET_REPLACE_INTERVAL', '1000' if OGBENCH_DATASET_DIR else '0'))

PROFILE_OVERRIDES = {
    'smoke': dict(
        run_group='notebook_report',
        tags='TRQAM_NOTEBOOK_SMOKE',
        seed=0,
        save_dir='exp_notebook',
        offline_steps=1,
        online_steps=0,
        log_interval=1,
        eval_interval=1,
        save_interval=0,
        eval_episodes=1,
        dataset_proportion=0.001,
        dataset_replace_interval=0,
        start_training=5,
        bc_only=True,
        batch_size=8,
        actor_hidden_dims=(64, 64),
        value_hidden_dims=(64, 64),
        actor_layer_norm=False,
        flow_steps=2,
    ),
    'full_bc': dict(
        run_group='bc_pretrain',
        tags='BC_FULL',
        seed=10001,
        save_dir='exp_full',
        ogbench_dataset_dir=OGBENCH_DATASET_DIR,
        offline_steps=300_000,
        online_steps=0,
        log_interval=5_000,
        eval_interval=50_000,
        save_interval=50_000,
        eval_episodes=50,
        dataset_proportion=1.0,
        dataset_replace_interval=FULL_DATASET_REPLACE_INTERVAL,
        start_training=5_000,
        bc_only=True,
        batch_size=256,
        actor_hidden_dims=FULL_HIDDEN_DIMS,
        value_hidden_dims=FULL_HIDDEN_DIMS,
        actor_layer_norm=FULL_ACTOR_LAYER_NORM,
        flow_steps=10,
    ),
    'full_trqam': dict(
        run_group='reproduce',
        tags='TRQAM_FULL',
        seed=10001,
        save_dir='exp_full',
        ogbench_dataset_dir=OGBENCH_DATASET_DIR,
        pretrained_actor_path=PRETRAINED_ACTOR_PATH,
        offline_steps=1_000_000,
        online_steps=500_000,
        log_interval=5_000,
        eval_interval=50_000,
        save_interval=50_000,
        eval_episodes=50,
        dataset_proportion=1.0,
        dataset_replace_interval=FULL_DATASET_REPLACE_INTERVAL,
        start_training=5_000,
        bc_only=False,
        kl_budget=0.5,
        batch_size=256,
        actor_hidden_dims=FULL_HIDDEN_DIMS,
        value_hidden_dims=FULL_HIDDEN_DIMS,
        actor_layer_norm=FULL_ACTOR_LAYER_NORM,
        flow_steps=10,
    ),
    'full_pipeline': dict(
        run_group='reproduce',
        tags='TRQAM_FULL',
        seed=10001,
        save_dir='exp_full',
        ogbench_dataset_dir=OGBENCH_DATASET_DIR,
        pretrained_actor_path=None,
        offline_steps=1_000_000,
        online_steps=500_000,
        log_interval=5_000,
        eval_interval=50_000,
        save_interval=50_000,
        eval_episodes=50,
        dataset_proportion=1.0,
        dataset_replace_interval=FULL_DATASET_REPLACE_INTERVAL,
        start_training=5_000,
        bc_only=False,
        kl_budget=0.5,
        batch_size=256,
        actor_hidden_dims=FULL_HIDDEN_DIMS,
        value_hidden_dims=FULL_HIDDEN_DIMS,
        actor_layer_norm=FULL_ACTOR_LAYER_NORM,
        flow_steps=10,
    ),
}

if RUN_PROFILE not in PROFILE_OVERRIDES:
    raise ValueError(f'Unknown RUN_PROFILE={RUN_PROFILE!r}. Choose one of {sorted(PROFILE_OVERRIDES)}')

cfg = NotebookConfig(**PROFILE_OVERRIDES[RUN_PROFILE])

if RUN_PROFILE == 'full_trqam' and EXECUTION_MODE == 'notebook' and not cfg.pretrained_actor_path:
    raise RuntimeError('full_trqam notebook execution requires PRETRAINED_ACTOR_PATH.')

config_rows = [
    ('RUN_PROFILE', RUN_PROFILE),
    ('EXECUTION_MODE', EXECUTION_MODE),
    ('SUBMIT_SLURM_JOB', SUBMIT_SLURM_JOB),
    ('WAIT_FOR_SLURM_COMPLETION', WAIT_FOR_SLURM_COMPLETION),
    ('SLURM_PARTITION', SLURM_PARTITION),
    ('SLURM_TIME', SLURM_TIME),
] + [(key, fmt_value(value)) for key, value in asdict(cfg).items()]
display_kv('Experiment Configuration', config_rows)


In [ ]:
def build_agent_config(cfg: NotebookConfig):
    config = get_trqam_config()
    config['horizon_length'] = cfg.horizon_length
    config['action_chunking'] = cfg.action_chunking
    config['bc_only'] = cfg.bc_only
    config['kl_budget'] = cfg.kl_budget
    config['batch_size'] = cfg.batch_size
    config['actor_hidden_dims'] = tuple(cfg.actor_hidden_dims)
    config['value_hidden_dims'] = tuple(cfg.value_hidden_dims)
    config['actor_layer_norm'] = cfg.actor_layer_norm
    config['flow_steps'] = cfg.flow_steps

    if cfg.large_network:
        config['actor_hidden_dims'] = (1024, 1024, 1024, 1024)
        config['value_hidden_dims'] = (1024, 1024, 1024, 1024)
        config['actor_layer_norm'] = True

    return config


agent_config = build_agent_config(cfg)
agent_config_rows = [
    ('agent_name', agent_config['agent_name']),
    ('batch_size', agent_config['batch_size']),
    ('actor_hidden_dims', fmt_value(agent_config['actor_hidden_dims'])),
    ('value_hidden_dims', fmt_value(agent_config['value_hidden_dims'])),
    ('actor_layer_norm', agent_config['actor_layer_norm']),
    ('flow_steps', agent_config['flow_steps']),
    ('horizon_length', agent_config['horizon_length']),
    ('action_chunking', agent_config['action_chunking']),
    ('bc_only', agent_config['bc_only']),
    ('kl_budget', agent_config['kl_budget']),
    ('discount', agent_config['discount']),
]
display_kv('Agent Configuration', agent_config_rows)


## 4. Slurm Full-Run Controller

This section keeps the Slurm workflow inside the notebook. It writes generated scripts to `slurm_notebook/`, submits the selected job or pipeline, and records job IDs in `slurm_notebook/latest_jobs.json`.

For the default `full_pipeline` profile, Run All submits two jobs: BC pretraining first and TRQAM fine-tuning second with an `afterok:<BC_JOB_ID>` dependency. The final training report is generated by the Slurm job after CSV logs are written.


In [ ]:
import subprocess

SLURM_NOTEBOOK_DIR = ROOT / 'slurm_notebook'
SLURM_NOTEBOOK_DIR.mkdir(exist_ok=True)

EXPORT_EVAL_REPORT_SOURCE = "import argparse\nimport glob\nimport html\nimport json\nimport time\nfrom pathlib import Path\n\nimport matplotlib.pyplot as plt\nimport pandas as pd\n\n\nCSS = \"\"\"\n<style>\nbody { color: #1f2328; font-family: -apple-system, BlinkMacSystemFont, \"Segoe UI\", sans-serif; }\n.muted { color: #57606a; }\n.callout { border-left: 4px solid #0969da; background: #f6f8fa; padding: 12px 14px; margin: 12px 0 18px; }\n.grid { display: grid; grid-template-columns: repeat(auto-fit, minmax(150px, 1fr)); gap: 10px; margin: 12px 0 18px; }\n.kpi { border: 1px solid #d0d7de; border-radius: 6px; padding: 10px 12px; }\n.kpi .label { color: #57606a; font-size: 12px; margin-bottom: 4px; }\n.kpi .value { font-weight: 650; font-size: 18px; overflow-wrap: anywhere; }\ntable { border-collapse: collapse; width: 100%; font-size: 13px; }\nth, td { border: 1px solid #d0d7de; padding: 6px 8px; text-align: left; vertical-align: top; }\nth { background: #f6f8fa; }\nimg { max-width: 100%; border: 1px solid #d0d7de; border-radius: 6px; }\n</style>\n\"\"\"\n\n\ndef parse_args():\n    parser = argparse.ArgumentParser(description=\"Export a compact TRQAM evaluation report from CSV logs.\")\n    parser.add_argument(\"--run-dir\", type=Path, default=None, help=\"Specific run directory containing eval.csv.\")\n    parser.add_argument(\"--latest-glob\", default=None, help=\"Glob pattern for run directories; newest directory is used.\")\n    parser.add_argument(\"--output-name\", default=\"report.html\", help=\"Report filename to write inside the run directory.\")\n    return parser.parse_args()\n\n\ndef resolve_run_dir(run_dir, latest_glob):\n    if run_dir is not None:\n        if not run_dir.is_dir():\n            raise FileNotFoundError(f\"Run directory does not exist: {run_dir}\")\n        return run_dir\n\n    if not latest_glob:\n        raise ValueError(\"Provide either --run-dir or --latest-glob\")\n\n    candidates = [Path(path) for path in glob.glob(latest_glob) if Path(path).is_dir()]\n    candidates = [path for path in candidates if (path / \"eval.csv\").exists() or (path / \"flags.json\").exists()]\n    if not candidates:\n        raise FileNotFoundError(f\"No run directories matched: {latest_glob}\")\n    return max(candidates, key=lambda path: path.stat().st_mtime)\n\n\ndef read_csv(path):\n    return pd.read_csv(path) if path.exists() else pd.DataFrame()\n\n\ndef read_flags(path):\n    if not path.exists():\n        return {}\n    with path.open() as f:\n        return json.load(f)\n\n\ndef fmt(value):\n    if isinstance(value, float):\n        return f\"{value:.6g}\"\n    return str(value)\n\n\ndef last_value(df, key, default=\"n/a\"):\n    if df.empty or key not in df.columns:\n        return default\n    return fmt(df.iloc[-1][key])\n\n\ndef table_html(df, empty_message, max_rows=5):\n    if df.empty:\n        return f'<p class=\"muted\">{html.escape(empty_message)}</p>'\n    return df.tail(max_rows).to_html(index=False, border=0, escape=True, float_format=lambda x: f\"{x:.6g}\")\n\n\ndef card(label, value):\n    return (\n        f'<div class=\"kpi\"><div class=\"label\">{html.escape(str(label))}</div>'\n        f'<div class=\"value\">{html.escape(str(value))}</div></div>'\n    )\n\n\ndef plot_metrics(df, title, candidates, out_dir):\n    metrics = [\n        metric\n        for metric in candidates\n        if metric in df.columns and pd.api.types.is_numeric_dtype(df[metric])\n    ]\n    if df.empty or not metrics or \"step\" not in df.columns:\n        return None\n\n    fig, ax = plt.subplots(figsize=(9, 4))\n    df.plot(x=\"step\", y=metrics, marker=\"o\", ax=ax)\n    ax.set_title(title)\n    ax.set_xlabel(\"step\")\n    ax.grid(True, alpha=0.3)\n    fig.tight_layout()\n\n    filename = title.lower().replace(\" \", \"_\").replace(\"/\", \"_\") + \".png\"\n    path = out_dir / filename\n    fig.savefig(path, dpi=160, bbox_inches=\"tight\")\n    plt.close(fig)\n    return path\n\n\ndef main():\n    args = parse_args()\n    run_dir = resolve_run_dir(args.run_dir, args.latest_glob)\n\n    eval_df = read_csv(run_dir / \"eval.csv\")\n    offline_df = read_csv(run_dir / \"offline_agent.csv\")\n    online_df = read_csv(run_dir / \"online_agent.csv\")\n    flags = read_flags(run_dir / \"flags.json\")\n\n    fig_dir = run_dir / \"figures\"\n    fig_dir.mkdir(exist_ok=True)\n    figures = [\n        plot_metrics(eval_df, \"Evaluation metrics\", [\"success\", \"episode.return\", \"episode.length\", \"episode.final_reward\"], fig_dir),\n        plot_metrics(offline_df, \"Offline agent diagnostics\", [\"actor/flow_loss\", \"actor/fast_loss\", \"actor/path_kl\", \"critic/critic_loss\", \"dual/lambda\"], fig_dir),\n        plot_metrics(online_df, \"Online agent diagnostics\", [\"actor/flow_loss\", \"actor/fast_loss\", \"actor/path_kl\", \"critic/critic_loss\", \"dual/lambda\"], fig_dir),\n    ]\n    figures = [path for path in figures if path is not None]\n\n    cards = \"\".join(\n        [\n            card(\"Environment\", flags.get(\"env_name\", run_dir.parent.name)),\n            card(\"Seed\", flags.get(\"seed\", \"n/a\")),\n            card(\"Offline steps\", flags.get(\"offline_steps\", \"n/a\")),\n            card(\"Online steps\", flags.get(\"online_steps\", \"n/a\")),\n            card(\"Success\", last_value(eval_df, \"success\")),\n            card(\"Return\", last_value(eval_df, \"episode.return\")),\n            card(\"Episode length\", last_value(eval_df, \"episode.length\")),\n            card(\"Final eval step\", last_value(eval_df, \"step\")),\n        ]\n    )\n\n    flags_table = pd.DataFrame(sorted(flags.items()), columns=[\"Item\", \"Value\"]).to_html(\n        index=False,\n        border=0,\n        escape=True,\n    )\n\n    figure_html = \"\"\n    for figure in figures:\n        rel = figure.relative_to(run_dir).as_posix()\n        title = figure.stem.replace(\"_\", \" \").title()\n        figure_html += f\"<h3>{html.escape(title)}</h3><img src=\\\"{html.escape(rel)}\\\" alt=\\\"{html.escape(title)}\\\">\"\n    if not figure_html:\n        figure_html = '<p class=\"muted\">No figures were generated.</p>'\n\n    body = f\"\"\"\n<h1>TRQAM Evaluation Report</h1>\n<p class=\"muted\">Generated at: {html.escape(time.strftime(\"%Y-%m-%d %H:%M:%S\"))}</p>\n<div class=\"callout\">This report was generated from CSV logs in <code>{html.escape(str(run_dir))}</code>.</div>\n<h2>Key Metrics</h2>\n<div class=\"grid\">{cards}</div>\n<h2>Final Evaluation Log</h2>\n{table_html(eval_df, \"No evaluation logs.\", max_rows=1)}\n<h2>Recent Offline Logs</h2>\n{table_html(offline_df, \"No offline logs.\", max_rows=5)}\n<h2>Recent Online Logs</h2>\n{table_html(online_df, \"No online logs.\", max_rows=5)}\n<h2>Configuration</h2>\n{flags_table}\n<h2>Figures</h2>\n{figure_html}\n\"\"\"\n\n    report = f\"<!doctype html><html lang=\\\"en\\\"><head><meta charset=\\\"utf-8\\\"><title>TRQAM Evaluation Report</title>{CSS}</head><body style=\\\"max-width: 980px; margin: 32px auto; padding: 0 20px;\\\">{body}</body></html>\"\n    report_path = run_dir / args.output_name\n    report_path.write_text(report, encoding=\"utf-8\")\n\n    print(f\"run_dir={run_dir}\")\n    print(f\"report={report_path}\")\n    if not eval_df.empty:\n        print(\"final_eval=\" + eval_df.tail(1).to_json(orient=\"records\"))\n\n\nif __name__ == \"__main__\":\n    main()\n"
FULL_BC_SBATCH_SOURCE = "#!/bin/bash\n#SBATCH --job-name=trqam-full-bc\n#SBATCH --partition=4090\n#SBATCH --gres=gpu:1\n#SBATCH --cpus-per-task=8\n#SBATCH --mem=64G\n#SBATCH --time=3-00:00:00\n#SBATCH --output=slurm_logs/trqam-full-bc-%j.out\n#SBATCH --error=slurm_logs/trqam-full-bc-%j.err\n\nset -euo pipefail\n\ncd /mnt/lustre/slurm/users/sanghyeok/trqam\nmkdir -p slurm_logs\n\nexport PYTHONPATH=/mnt/lustre/slurm/users/sanghyeok/trqam:${PYTHONPATH:-}\nexport WANDB_MODE=${WANDB_MODE:-disabled}\nexport MUJOCO_GL=egl\nexport XLA_PYTHON_CLIENT_PREALLOCATE=false\n\nPYTHON=/home/sanghyeok/miniconda3/envs/trqam/bin/python\n\nENV_NAME=${ENV_NAME:-cube-triple-play-singletask-task2-v0}\nSEED=${SEED:-10001}\nSAVE_ROOT=${SAVE_ROOT:-exp_full}\nRUN_GROUP=${RUN_GROUP:-bc_pretrain}\nTAGS=${TAGS:-BC_FULL}\nHORIZON_LENGTH=${HORIZON_LENGTH:-5}\nWIDTH=${WIDTH:-512}\nACTOR_LAYER_NORM=${ACTOR_LAYER_NORM:-False}\nFLOW_STEPS=${FLOW_STEPS:-10}\nBATCH_SIZE=${BATCH_SIZE:-256}\nEVAL_EPISODES=${EVAL_EPISODES:-50}\nLOG_INTERVAL=${LOG_INTERVAL:-5000}\nEVAL_INTERVAL=${EVAL_INTERVAL:-50000}\nSAVE_INTERVAL=${SAVE_INTERVAL:-50000}\nDATASET_PROPORTION=${DATASET_PROPORTION:-1.0}\n\nHIDDEN_DIMS=\"(${WIDTH},${WIDTH},${WIDTH},${WIDTH})\"\n\nDATASET_ARGS=()\nif [[ -n \"${OGBENCH_DATASET_DIR:-}\" ]]; then\n  DATASET_ARGS+=(\"--ogbench_dataset_dir=${OGBENCH_DATASET_DIR}\")\n  DATASET_ARGS+=(\"--dataset_replace_interval=${DATASET_REPLACE_INTERVAL:-1000}\")\nelse\n  DATASET_ARGS+=(\"--dataset_replace_interval=0\")\nfi\n\nARGS=(\n  \"--save_dir=${SAVE_ROOT%/}/\"\n  \"--run_group=${RUN_GROUP}\"\n  \"--agent=agents/trqam.py\"\n  \"--tags=${TAGS}\"\n  \"--seed=${SEED}\"\n  \"--env_name=${ENV_NAME}\"\n  \"--sparse=False\"\n  \"--horizon_length=${HORIZON_LENGTH}\"\n  \"--agent.action_chunking=True\"\n  \"--bc_only=True\"\n  \"--offline_steps=300000\"\n  \"--online_steps=0\"\n  \"--dataset_proportion=${DATASET_PROPORTION}\"\n  \"--eval_interval=${EVAL_INTERVAL}\"\n  \"--eval_episodes=${EVAL_EPISODES}\"\n  \"--video_episodes=0\"\n  \"--save_interval=${SAVE_INTERVAL}\"\n  \"--log_interval=${LOG_INTERVAL}\"\n  \"--agent.batch_size=${BATCH_SIZE}\"\n  \"--agent.actor_hidden_dims=${HIDDEN_DIMS}\"\n  \"--agent.value_hidden_dims=${HIDDEN_DIMS}\"\n  \"--agent.actor_layer_norm=${ACTOR_LAYER_NORM}\"\n  \"--agent.flow_steps=${FLOW_STEPS}\"\n  \"--save_last_checkpoint=True\"\n)\n\n\"${PYTHON}\" main.py \"${ARGS[@]}\" \"${DATASET_ARGS[@]}\"\n\"${PYTHON}\" slurm/export_eval_report.py --latest-glob \"${SAVE_ROOT%/}/*/${RUN_GROUP}/${ENV_NAME}/*\"\n"
FULL_TRQAM_SBATCH_SOURCE = "#!/bin/bash\n#SBATCH --job-name=trqam-full\n#SBATCH --partition=4090\n#SBATCH --gres=gpu:1\n#SBATCH --cpus-per-task=8\n#SBATCH --mem=64G\n#SBATCH --time=3-00:00:00\n#SBATCH --output=slurm_logs/trqam-full-%j.out\n#SBATCH --error=slurm_logs/trqam-full-%j.err\n\nset -euo pipefail\n\ncd /mnt/lustre/slurm/users/sanghyeok/trqam\nmkdir -p slurm_logs\n\nexport PYTHONPATH=/mnt/lustre/slurm/users/sanghyeok/trqam:${PYTHONPATH:-}\nexport WANDB_MODE=${WANDB_MODE:-disabled}\nexport MUJOCO_GL=egl\nexport XLA_PYTHON_CLIENT_PREALLOCATE=false\n\nPYTHON=/home/sanghyeok/miniconda3/envs/trqam/bin/python\n\nENV_NAME=${ENV_NAME:-cube-triple-play-singletask-task2-v0}\nSEED=${SEED:-10001}\nSAVE_ROOT=${SAVE_ROOT:-exp_full}\nRUN_GROUP=${RUN_GROUP:-reproduce}\nTAGS=${TAGS:-TRQAM_FULL}\nHORIZON_LENGTH=${HORIZON_LENGTH:-5}\nWIDTH=${WIDTH:-512}\nACTOR_LAYER_NORM=${ACTOR_LAYER_NORM:-False}\nFLOW_STEPS=${FLOW_STEPS:-10}\nBATCH_SIZE=${BATCH_SIZE:-256}\nKL_BUDGET=${KL_BUDGET:-0.5}\nEVAL_EPISODES=${EVAL_EPISODES:-50}\nLOG_INTERVAL=${LOG_INTERVAL:-5000}\nEVAL_INTERVAL=${EVAL_INTERVAL:-50000}\nSAVE_INTERVAL=${SAVE_INTERVAL:-50000}\nSTART_TRAINING=${START_TRAINING:-5000}\nDATASET_PROPORTION=${DATASET_PROPORTION:-1.0}\n\nHIDDEN_DIMS=\"(${WIDTH},${WIDTH},${WIDTH},${WIDTH})\"\n\nif [[ -z \"${BC_CKPT:-}\" ]]; then\n  if [[ -d \"${SAVE_ROOT%/}\" ]]; then\n    BC_CKPT=$(find \"${SAVE_ROOT%/}\" -path \"*/bc_pretrain/${ENV_NAME}/*/params_300000.pkl\" -printf '%T@ %p\\n' 2>/dev/null | sort -n | tail -1 | cut -d' ' -f2-)\n  else\n    BC_CKPT=\"\"\n  fi\nfi\n\nif [[ -z \"${BC_CKPT:-}\" || ! -f \"${BC_CKPT}\" ]]; then\n  echo \"BC_CKPT is required. Run slurm/ogbench_full_bc.sbatch first or pass BC_CKPT=/path/to/params_300000.pkl.\" >&2\n  exit 2\nfi\n\nDATASET_ARGS=()\nif [[ -n \"${OGBENCH_DATASET_DIR:-}\" ]]; then\n  DATASET_ARGS+=(\"--ogbench_dataset_dir=${OGBENCH_DATASET_DIR}\")\n  DATASET_ARGS+=(\"--dataset_replace_interval=${DATASET_REPLACE_INTERVAL:-1000}\")\nelse\n  DATASET_ARGS+=(\"--dataset_replace_interval=0\")\nfi\n\nARGS=(\n  \"--save_dir=${SAVE_ROOT%/}/\"\n  \"--run_group=${RUN_GROUP}\"\n  \"--agent=agents/trqam.py\"\n  \"--tags=${TAGS}\"\n  \"--seed=${SEED}\"\n  \"--env_name=${ENV_NAME}\"\n  \"--sparse=False\"\n  \"--horizon_length=${HORIZON_LENGTH}\"\n  \"--agent.action_chunking=True\"\n  \"--pretrained_actor_path=${BC_CKPT}\"\n  \"--bc_only=False\"\n  \"--offline_steps=1000000\"\n  \"--online_steps=500000\"\n  \"--start_training=${START_TRAINING}\"\n  \"--dataset_proportion=${DATASET_PROPORTION}\"\n  \"--eval_interval=${EVAL_INTERVAL}\"\n  \"--eval_episodes=${EVAL_EPISODES}\"\n  \"--video_episodes=0\"\n  \"--save_interval=${SAVE_INTERVAL}\"\n  \"--log_interval=${LOG_INTERVAL}\"\n  \"--agent.batch_size=${BATCH_SIZE}\"\n  \"--agent.actor_hidden_dims=${HIDDEN_DIMS}\"\n  \"--agent.value_hidden_dims=${HIDDEN_DIMS}\"\n  \"--agent.actor_layer_norm=${ACTOR_LAYER_NORM}\"\n  \"--agent.flow_steps=${FLOW_STEPS}\"\n  \"--save_last_checkpoint=True\"\n  \"--agent.kl_budget=${KL_BUDGET}\"\n)\n\n\"${PYTHON}\" main.py \"${ARGS[@]}\" \"${DATASET_ARGS[@]}\"\n\"${PYTHON}\" slurm/export_eval_report.py --latest-glob \"${SAVE_ROOT%/}/*/${RUN_GROUP}/${ENV_NAME}/*\"\n"


def patch_sbatch_source(source, job_name):
    text = source
    text = text.replace('/mnt/lustre/slurm/users/sanghyeok/trqam', str(ROOT))
    text = text.replace('slurm/export_eval_report.py', str(SLURM_NOTEBOOK_DIR / 'export_eval_report.py'))
    text = text.replace('#SBATCH --partition=4090', f'#SBATCH --partition={SLURM_PARTITION}')
    text = text.replace('#SBATCH --gres=gpu:1', f'#SBATCH --gres={SLURM_GPUS}')
    text = text.replace('#SBATCH --cpus-per-task=8', f'#SBATCH --cpus-per-task={SLURM_CPUS}')
    text = text.replace('#SBATCH --mem=64G', f'#SBATCH --mem={SLURM_MEM}')
    text = text.replace('#SBATCH --time=3-00:00:00', f'#SBATCH --time={SLURM_TIME}')
    text = text.replace('#SBATCH --job-name=trqam-full-bc', f'#SBATCH --job-name={job_name}')
    text = text.replace('#SBATCH --job-name=trqam-full', f'#SBATCH --job-name={job_name}')
    text = text.replace('trqam-full-bc-%j', f'{job_name}-%j')
    text = text.replace('trqam-full-%j', f'{job_name}-%j')
    return text


def write_slurm_files():
    exporter_path = SLURM_NOTEBOOK_DIR / 'export_eval_report.py'
    bc_path = SLURM_NOTEBOOK_DIR / 'full_bc.sbatch'
    trqam_path = SLURM_NOTEBOOK_DIR / 'full_trqam.sbatch'

    exporter_path.write_text(EXPORT_EVAL_REPORT_SOURCE, encoding='utf-8')
    bc_path.write_text(patch_sbatch_source(FULL_BC_SBATCH_SOURCE, 'trqam-nb-bc'), encoding='utf-8')
    trqam_path.write_text(patch_sbatch_source(FULL_TRQAM_SBATCH_SOURCE, 'trqam-nb-full'), encoding='utf-8')
    return {'bc': bc_path, 'trqam': trqam_path, 'exporter': exporter_path}


def slurm_export_env(stage):
    values = {
        'ENV_NAME': cfg.env_name,
        'SEED': cfg.seed,
        'SAVE_ROOT': cfg.save_dir,
        'HORIZON_LENGTH': cfg.horizon_length,
        'WIDTH': FULL_WIDTH,
        'ACTOR_LAYER_NORM': 'True' if FULL_ACTOR_LAYER_NORM else 'False',
        'FLOW_STEPS': cfg.flow_steps,
        'BATCH_SIZE': cfg.batch_size,
        'EVAL_EPISODES': cfg.eval_episodes,
        'LOG_INTERVAL': cfg.log_interval,
        'EVAL_INTERVAL': cfg.eval_interval,
        'SAVE_INTERVAL': cfg.save_interval,
        'DATASET_PROPORTION': cfg.dataset_proportion,
        'DATASET_REPLACE_INTERVAL': cfg.dataset_replace_interval,
    }
    if cfg.ogbench_dataset_dir:
        values['OGBENCH_DATASET_DIR'] = cfg.ogbench_dataset_dir
    if stage == 'bc':
        values['RUN_GROUP'] = 'bc_pretrain'
        values['TAGS'] = 'BC_FULL'
    else:
        values['RUN_GROUP'] = 'reproduce'
        values['TAGS'] = 'TRQAM_FULL'
        values['START_TRAINING'] = cfg.start_training
        values['KL_BUDGET'] = cfg.kl_budget
        if cfg.pretrained_actor_path:
            values['BC_CKPT'] = cfg.pretrained_actor_path
    return values


def submit_sbatch(script_path, stage, dependency=None):
    export_values = slurm_export_env(stage)
    export_arg = 'ALL,' + ','.join(f'{key}={value}' for key, value in export_values.items())
    cmd = ['sbatch', f'--export={export_arg}']
    if dependency:
        cmd.append(f'--dependency=afterok:{dependency}')
    cmd.append(str(script_path))
    result = subprocess.run(cmd, cwd=ROOT, text=True, capture_output=True, check=True)
    output = result.stdout.strip()
    job_id = output.split()[-1]
    return job_id, output


def slurm_status(job_id):
    result = subprocess.run(
        ['sacct', '-j', str(job_id), '-o', 'JobID,JobName,State,ExitCode,Elapsed,NodeList'],
        cwd=ROOT,
        text=True,
        capture_output=True,
        check=False,
    )
    return result.stdout.strip() or result.stderr.strip()


SLURM_JOB_IDS = {}
SLURM_FILES = {}

if EXECUTION_MODE == 'slurm' and RUN_PROFILE in {'full_bc', 'full_trqam', 'full_pipeline'}:
    SLURM_FILES = write_slurm_files()
    display_kv('Generated Slurm Files', [(name, f'`{path}`') for name, path in SLURM_FILES.items()])

    if SUBMIT_SLURM_JOB:
        if RUN_PROFILE == 'full_bc':
            job_id, output = submit_sbatch(SLURM_FILES['bc'], 'bc')
            SLURM_JOB_IDS['bc'] = job_id
            display(Markdown(f'BC job submitted: `{output}`'))
        elif RUN_PROFILE == 'full_trqam':
            job_id, output = submit_sbatch(SLURM_FILES['trqam'], 'trqam')
            SLURM_JOB_IDS['trqam'] = job_id
            display(Markdown(f'TRQAM job submitted: `{output}`'))
        else:
            bc_job, bc_output = submit_sbatch(SLURM_FILES['bc'], 'bc')
            trqam_job, trqam_output = submit_sbatch(SLURM_FILES['trqam'], 'trqam', dependency=bc_job)
            SLURM_JOB_IDS['bc'] = bc_job
            SLURM_JOB_IDS['trqam'] = trqam_job
            display(Markdown(f'BC job submitted: `{bc_output}`'))
            display(Markdown(f'TRQAM job submitted with dependency `afterok:{bc_job}`: `{trqam_output}`'))

        latest_jobs_path = SLURM_NOTEBOOK_DIR / 'latest_jobs.json'
        latest_jobs_path.write_text(json.dumps(SLURM_JOB_IDS, indent=2), encoding='utf-8')
        display_kv('Submitted Slurm Jobs', list(SLURM_JOB_IDS.items()))
    else:
        display(Markdown('Slurm submission is disabled because `SUBMIT_SLURM_JOB=False`.'))
else:
    display(Markdown('Slurm controller skipped because this run is configured for notebook-local execution.'))


## 4. Dataset and Environment Loading

The notebook follows the same dataset path as `main.py`. If `ogbench_dataset_dir` is set, it loads custom `.npz` shards from that directory. Otherwise it uses the default OGBench loader for `env_name`. Reward shaping, sparse reward conversion, and RoboMimic reward correction follow the original training code.


In [ ]:
def load_env_and_datasets_for_notebook(cfg: NotebookConfig):
    dataset_paths = []

    if cfg.ogbench_dataset_dir is not None:
        dataset_paths = [
            path for path in sorted(glob.glob(str(Path(cfg.ogbench_dataset_dir).expanduser() / '*.npz')))
            if '-val.npz' not in path
        ]
        if cfg.dataset_proportion < 1.0:
            subset_size = max(1, int(len(dataset_paths) * cfg.dataset_proportion))
            dataset_paths = dataset_paths[:subset_size]
        if not dataset_paths:
            raise FileNotFoundError(f'No training npz files found in {cfg.ogbench_dataset_dir}')

        env, eval_env, train_dataset, val_dataset = make_ogbench_env_and_datasets(
            cfg.env_name,
            dataset_path=dataset_paths[0],
            compact_dataset=False,
        )
    else:
        env, eval_env, train_dataset, val_dataset = make_env_and_datasets(cfg.env_name)

    return env, eval_env, train_dataset, val_dataset, dataset_paths


def process_train_dataset(ds, cfg: NotebookConfig):
    ds = Dataset.create(**ds)

    if cfg.dataset_proportion < 1.0 and cfg.ogbench_dataset_dir is None:
        new_size = int(len(ds['masks']) * cfg.dataset_proportion)
        ds = Dataset.create(**{key: value[:new_size] for key, value in ds.items()})

    if is_robomimic_env(cfg.env_name):
        ds_dict = {key: value for key, value in ds.items()}
        ds_dict['rewards'] = ds['rewards'] - 1.0
        ds = Dataset.create(**ds_dict)

    if cfg.sparse:
        ds_dict = {key: value for key, value in ds.items()}
        ds_dict['rewards'] = (ds['rewards'] != 0.0) * -1.0
        ds = Dataset.create(**ds_dict)

    return ds


if EXECUTION_MODE == 'notebook':
    env, eval_env, train_dataset, val_dataset, dataset_paths = load_env_and_datasets_for_notebook(cfg)
    train_dataset = process_train_dataset(train_dataset, cfg)
    example_batch = train_dataset.sample(())

    dataset_rows = [
        ('Environment', cfg.env_name),
        ('Train transitions', f'{train_dataset.size:,}'),
        ('Observation shape', example_batch['observations'].shape),
        ('Action shape', example_batch['actions'].shape),
        ('Dataset proportion', cfg.dataset_proportion),
        ('Custom dataset files', len(dataset_paths)),
        ('Dataset replace interval', cfg.dataset_replace_interval),
        ('Sparse reward', cfg.sparse),
    ]
    display_kv('Dataset and Environment Summary', dataset_rows)
else:
    env = eval_env = train_dataset = val_dataset = example_batch = None
    dataset_paths = []
    display(Markdown('Dataset loading is skipped in Slurm mode. The generated Slurm job loads the dataset on the compute node.'))


## 5. Agent Initialization

The TRQAM agent contains a critic ensemble, a slow flow actor, a fast flow actor, and target networks. When `pretrained_actor_path` is provided, the BC-trained flow policy is copied into the actor parameters used by the fine-tuning agent.


In [ ]:
def load_pretrained_actor(agent, checkpoint_path: str, config):
    checkpoint_path = str(Path(checkpoint_path).expanduser())
    with open(checkpoint_path, 'rb') as f:
        pretrained_dict = pickle.load(f)

    pretrained_params = pretrained_dict['agent']['network']['params']
    current_params = agent.network.params
    new_params = dict(current_params)

    pretrained_flow = None
    for key in ['modules_actor_slow', 'modules_actor_bc_flow', 'modules_actor_flow', 'modules_actor']:
        if key in pretrained_params:
            pretrained_flow = pretrained_params[key]
            display(Markdown(f'- Found pretrained flow policy: `{key}`'))
            break

    if pretrained_flow is None:
        display(Markdown('- Warning: no flow policy found in checkpoint.'))
        return agent

    loaded = False
    if 'modules_actor_slow' in current_params:
        new_params['modules_actor_slow'] = pretrained_flow
        new_params['modules_target_actor_slow'] = pretrained_flow
        if 'modules_actor_fast' in current_params:
            new_params['modules_actor_fast'] = pretrained_flow
            new_params['modules_target_actor_fast'] = pretrained_flow
        loaded = True

    if 'modules_actor_bc_flow' in current_params:
        new_params['modules_actor_bc_flow'] = pretrained_flow
        if 'modules_target_actor_bc_flow' in current_params:
            new_params['modules_target_actor_bc_flow'] = pretrained_flow
        loaded = True

    if 'modules_actor' in current_params and config['agent_name'] in ['cgql', 'ifql']:
        new_params['modules_actor'] = pretrained_flow
        if 'modules_target_actor' in current_params:
            new_params['modules_target_actor'] = pretrained_flow
        loaded = True

    if loaded:
        display(Markdown(f'- Pretrained actor loaded: `{checkpoint_path}`'))
        return agent.replace(network=agent.network.replace(params=new_params))

    display(Markdown('- Warning: checkpoint could not be mapped to current agent.'))
    return agent


random.seed(cfg.seed)
np.random.seed(cfg.seed)

if EXECUTION_MODE == 'notebook':
    agent_class = agents[agent_config['agent_name']]
    agent = agent_class.create(
        cfg.seed,
        example_batch['observations'],
        example_batch['actions'],
        agent_config,
    )

    if cfg.pretrained_actor_path is not None:
        agent = load_pretrained_actor(agent, cfg.pretrained_actor_path, agent_config)

    params_without_targets = {key: value for key, value in agent.network.params.items() if 'target' not in key}
    param_count = sum(leaf.size for leaf in jax.tree_util.tree_leaves(params_without_targets))
    agent_rows = [
        ('Agent', agent_config['agent_name']),
        ('Parameters without target nets', f'{param_count:,}'),
        ('Pretrained actor', cfg.pretrained_actor_path or 'not used'),
        ('Action chunking', cfg.action_chunking),
        ('Horizon length', cfg.horizon_length),
    ]
    display_kv('Agent Initialization Result', agent_rows)
else:
    agent = None
    display(Markdown('Agent initialization is skipped in Slurm mode. The generated Slurm job initializes the agent on the compute node.'))


## 6. Training and Evaluation Functions

The following functions split the original `main.py` offline and online loops into notebook-executable units. Logs are written to CSV files as they are produced, so long full runs preserve intermediate evaluation results even before the final report cell executes.


In [ ]:
def make_save_dir(cfg: NotebookConfig, config):
    env_short = cfg.env_name.split('-')[0]
    timestamp = time.strftime('%Y%m%d-%H%M%S')
    exp_name = f'{cfg.tags}_{config["agent_name"]}_{env_short}_s{cfg.seed}_{timestamp}'
    save_dir = Path(cfg.save_dir) / cfg.run_group / cfg.env_name / exp_name
    save_dir.mkdir(parents=True, exist_ok=True)
    with open(save_dir / 'notebook_config.json', 'w') as f:
        json.dump(asdict(cfg), f, indent=2)
    return save_dir


def scalarize(row):
    out = {}
    for key, value in row.items():
        arr = np.asarray(value)
        out[key] = float(arr) if arr.shape == () else value
    return out


class NotebookLogger:
    def __init__(self, save_dir: Path):
        self.save_dir = save_dir
        self.rows = defaultdict(list)

    def log(self, data, prefix: str, step: int):
        row = scalarize(data)
        row['step'] = step
        self.rows[prefix].append(row)
        path = self.save_dir / f'{prefix}.csv'
        pd.DataFrame([row]).to_csv(path, mode='a', header=not path.exists(), index=False)

    def frame(self, prefix: str):
        return pd.DataFrame(self.rows[prefix])

    def save_all(self):
        for prefix in self.rows:
            self.frame(prefix).to_csv(self.save_dir / f'{prefix}.csv', index=False)


if EXECUTION_MODE == 'notebook':
    save_dir = make_save_dir(cfg, agent_config)
else:
    save_dir = SLURM_NOTEBOOK_DIR / 'controller_outputs' / RUN_PROFILE
    save_dir.mkdir(parents=True, exist_ok=True)

logger = NotebookLogger(save_dir)
report_figures = {}

display_kv('Output Path', [('Current notebook output directory', f'`{save_dir}`')])


In [ ]:
def maybe_replace_ogbench_dataset(step, cfg, dataset_paths, dataset_idx, env):
    if not dataset_paths or cfg.dataset_replace_interval == 0:
        return None, dataset_idx
    if step % cfg.dataset_replace_interval != 0:
        return None, dataset_idx

    dataset_idx = (dataset_idx + 1) % len(dataset_paths)
    train_dataset, _ = make_ogbench_env_and_datasets(
        cfg.env_name,
        dataset_path=dataset_paths[dataset_idx],
        compact_dataset=False,
        dataset_only=True,
        cur_env=env,
    )
    return process_train_dataset(train_dataset, cfg), dataset_idx


def run_offline_phase(agent, env, eval_env, train_dataset, example_batch, cfg, config, logger, save_dir, dataset_paths):
    if cfg.offline_steps <= 0:
        return agent, train_dataset

    dataset_idx = 0
    action_dim = example_batch['actions'].shape[-1]
    last_save_path = None

    for step in trange(1, cfg.offline_steps + 1, desc='offline'):
        new_dataset, dataset_idx = maybe_replace_ogbench_dataset(step, cfg, dataset_paths, dataset_idx, env)
        if new_dataset is not None:
            train_dataset = new_dataset

        batch = train_dataset.sample_sequence(
            config['batch_size'],
            sequence_length=cfg.horizon_length,
            discount=config['discount'],
        )
        agent, info = agent.update(batch)

        if step == 1 or step % cfg.log_interval == 0:
            logger.log(info, 'offline_agent', step=step)

        should_eval = step == cfg.offline_steps or (cfg.eval_interval > 0 and step % cfg.eval_interval == 0)
        if should_eval:
            eval_info, _, _ = evaluate(
                agent=agent,
                env=eval_env,
                action_dim=action_dim,
                num_eval_episodes=cfg.eval_episodes,
                num_video_episodes=cfg.video_episodes,
                video_frame_skip=cfg.video_frame_skip,
            )
            logger.log(eval_info, 'eval', step=step)

        if cfg.save_interval > 0 and step % cfg.save_interval == 0:
            last_save_path = save_agent(agent, str(save_dir), step)

    logger.save_all()
    return agent, train_dataset

In [ ]:
def run_online_phase(agent, env, eval_env, train_dataset, example_batch, cfg, config, logger, save_dir):
    if cfg.online_steps <= 0:
        return agent

    if cfg.kl_budget_online is not None:
        config['kl_budget'] = cfg.kl_budget_online
        new_config = dict(agent.config)
        new_config['kl_budget'] = cfg.kl_budget_online
        object.__setattr__(agent, 'config', flax.core.FrozenDict(new_config))

    action_dim = example_batch['actions'].shape[-1]
    if cfg.balanced_sampling:
        replay_buffer = ReplayBuffer.create(example_batch, size=cfg.online_steps)
    else:
        replay_buffer = ReplayBuffer.create_from_initial_dataset(
            dict(train_dataset),
            size=train_dataset.size + cfg.online_steps,
        )

    online_rng = jax.random.PRNGKey(cfg.seed + 1)
    observation, _ = env.reset()
    action_queue = []
    latest_update_info = None

    for step in trange(1, cfg.online_steps + 1, desc='online'):
        log_step = cfg.offline_steps + step
        online_rng, key = jax.random.split(online_rng)

        if not action_queue:
            if cfg.balanced_sampling and step < cfg.start_training:
                action = np.random.rand(action_dim) * 2.0 - 1.0
            else:
                action = agent.sample_actions(observations=observation, rng=key)
            for chunk_action in np.asarray(action).reshape(-1, action_dim):
                action_queue.append(chunk_action)

        action = np.clip(action_queue.pop(0), -1.0, 1.0)
        next_observation, reward, terminated, truncated, info = env.step(action)
        done = terminated or truncated

        env_info = {key: value for key, value in info.items() if key.startswith('distance')}
        if env_info:
            logger.log(env_info, 'env', step=log_step)

        if is_robomimic_env(cfg.env_name):
            reward = reward - 1.0
        if cfg.sparse:
            reward = (reward != 0.0) * -1.0

        replay_buffer.add_transition(
            dict(
                observations=observation,
                actions=action,
                rewards=reward,
                terminals=float(done),
                masks=1.0 - float(terminated),
                next_observations=next_observation,
            )
        )

        if done:
            observation, _ = env.reset()
            action_queue = []
        else:
            observation = next_observation

        if step >= cfg.start_training:
            if cfg.balanced_sampling:
                half_batch = config['batch_size'] // 2
                dataset_batch = train_dataset.sample_sequence(
                    half_batch * cfg.utd_ratio,
                    sequence_length=cfg.horizon_length,
                    discount=config['discount'],
                )
                replay_batch = replay_buffer.sample_sequence(
                    half_batch * cfg.utd_ratio,
                    sequence_length=cfg.horizon_length,
                    discount=config['discount'],
                )
                batch = {
                    key: np.concatenate(
                        [
                            dataset_batch[key].reshape((cfg.utd_ratio, half_batch) + dataset_batch[key].shape[1:]),
                            replay_batch[key].reshape((cfg.utd_ratio, half_batch) + replay_batch[key].shape[1:]),
                        ],
                        axis=1,
                    )
                    for key in dataset_batch
                }
            else:
                batch = replay_buffer.sample_sequence(
                    config['batch_size'] * cfg.utd_ratio,
                    sequence_length=cfg.horizon_length,
                    discount=config['discount'],
                )
                batch = jax.tree_util.tree_map(
                    lambda x: x.reshape((cfg.utd_ratio, config['batch_size']) + x.shape[1:]),
                    batch,
                )
            agent, latest_update_info = agent.batch_update(batch)

        if latest_update_info is not None and (step == cfg.start_training or step % cfg.log_interval == 0):
            logger.log(latest_update_info, 'online_agent', step=log_step)

        should_eval = step == cfg.online_steps or (cfg.eval_interval > 0 and step % cfg.eval_interval == 0)
        if should_eval:
            eval_info, _, _ = evaluate(
                agent=agent,
                env=eval_env,
                action_dim=action_dim,
                num_eval_episodes=cfg.eval_episodes,
                num_video_episodes=cfg.video_episodes,
                video_frame_skip=cfg.video_frame_skip,
            )
            logger.log(eval_info, 'eval', step=log_step)

        if cfg.save_interval > 0 and step % cfg.save_interval == 0:
            save_agent(agent, str(save_dir), log_step)

    logger.save_all()
    return agent

## 7. Experiment Execution

This cell runs the offline and online phases according to the selected profile. The first JAX compilation can take noticeable time. In full profiles, evaluation runs every `eval_interval` steps and checkpoints are saved every `save_interval` steps.


In [ ]:
if EXECUTION_MODE == 'notebook':
    agent, train_dataset = run_offline_phase(
        agent=agent,
        env=env,
        eval_env=eval_env,
        train_dataset=train_dataset,
        example_batch=example_batch,
        cfg=cfg,
        config=agent_config,
        logger=logger,
        save_dir=save_dir,
        dataset_paths=dataset_paths,
    )

    agent = run_online_phase(
        agent=agent,
        env=env,
        eval_env=eval_env,
        train_dataset=train_dataset,
        example_batch=example_batch,
        cfg=cfg,
        config=agent_config,
        logger=logger,
        save_dir=save_dir,
    )

    logger.save_all()
    log_files = sorted(path.name for path in save_dir.glob('*.csv'))
    display_kv(
        'Experiment Completed',
        [
            ('Saved logs', ', '.join(log_files) if log_files else 'no csv logs'),
            ('Save directory', f'`{save_dir}`'),
        ],
    )
else:
    display(Markdown('Notebook-local training is skipped. Full training has been submitted to Slurm from the controller section above.'))


## 8. Result Tables

Training logs are saved as `offline_agent.csv`, `online_agent.csv`, and `eval.csv`. The tables below show the final evaluation row and recent training diagnostics.


In [ ]:
def find_latest_slurm_run(run_group):
    patterns = [
        Path(cfg.save_dir).glob(f'*/{run_group}/{cfg.env_name}/*'),
        Path(cfg.save_dir).glob(f'{run_group}/{cfg.env_name}/*'),
    ]
    candidates = []
    for pattern in patterns:
        for path in pattern:
            if path.is_dir() and ((path / 'eval.csv').exists() or (path / 'flags.json').exists()):
                candidates.append(path)
    return max(candidates, key=lambda path: path.stat().st_mtime) if candidates else None


if EXECUTION_MODE == 'notebook':
    offline_df = logger.frame('offline_agent')
    online_df = logger.frame('online_agent')
    eval_df = logger.frame('eval')
else:
    if RUN_PROFILE == 'full_bc':
        candidate_run_dir = find_latest_slurm_run('bc_pretrain')
    else:
        candidate_run_dir = find_latest_slurm_run('reproduce') or find_latest_slurm_run('bc_pretrain')

    if candidate_run_dir is not None:
        save_dir = candidate_run_dir
        offline_df = pd.read_csv(save_dir / 'offline_agent.csv') if (save_dir / 'offline_agent.csv').exists() else pd.DataFrame()
        online_df = pd.read_csv(save_dir / 'online_agent.csv') if (save_dir / 'online_agent.csv').exists() else pd.DataFrame()
        eval_df = pd.read_csv(save_dir / 'eval.csv') if (save_dir / 'eval.csv').exists() else pd.DataFrame()
        display(Markdown(f'Loaded latest Slurm run directory: `{save_dir}`'))
    else:
        offline_df = pd.DataFrame()
        online_df = pd.DataFrame()
        eval_df = pd.DataFrame()
        display(Markdown('No completed Slurm result directory was found yet. Re-run the result/report cells after the Slurm job writes `eval.csv`.'))

display(Markdown('### Final Evaluation Result'))
if not eval_df.empty:
    display(eval_df.tail(1))
else:
    display(pd.DataFrame({'message': ['no eval logs']}))

display(Markdown('### Recent Offline Training Logs'))
if not offline_df.empty:
    display(offline_df.tail())
else:
    display(pd.DataFrame({'message': ['no offline logs']}))

if not online_df.empty:
    display(Markdown('### Recent Online Training Logs'))
    display(online_df.tail())


In [ ]:
def slugify(text):
    slug = re.sub(r'[^A-Za-z0-9]+', '_', text).strip('_').lower()
    return slug or 'figure'


def plot_metrics(df, title, metric_candidates):
    metrics = [
        metric for metric in metric_candidates
        if metric in df.columns and pd.api.types.is_numeric_dtype(df[metric])
    ]
    if df.empty or not metrics:
        display(Markdown(f'_No metrics to plot for **{title}**._'))
        return None

    fig, ax = plt.subplots(figsize=(9, 4))
    df.plot(x='step', y=metrics, marker='o', ax=ax)
    ax.set_title(title)
    ax.set_xlabel('step')
    ax.grid(True, alpha=0.3)
    fig.tight_layout()

    fig_dir = save_dir / 'figures'
    fig_dir.mkdir(parents=True, exist_ok=True)
    fig_path = fig_dir / f'{slugify(title)}.png'
    fig.savefig(fig_path, dpi=160, bbox_inches='tight')
    report_figures[title] = fig_path.relative_to(save_dir).as_posix()
    display(fig)
    plt.close(fig)
    return fig_path


plot_metrics(
    offline_df,
    'Offline agent diagnostics',
    ['actor/flow_loss', 'actor/fast_loss', 'actor/path_kl', 'critic/critic_loss', 'dual/lambda'],
)

plot_metrics(
    eval_df,
    'Evaluation metrics',
    [col for col in eval_df.columns if col != 'step'][:5],
)


## 9. Automatically Generated Report

The final cell renders a report block from the current run and writes the same content to `report.html` inside the run directory. After a full run, this section contains the configuration, final evaluation metrics, recent training diagnostics, and saved plots.


In [ ]:
def _format_float(value):
    if isinstance(value, float):
        return f'{value:.6g}'
    return value


def _df_to_report_html(df, empty_message='No records', max_rows=5):
    if df.empty:
        return f'<p class="trqam-muted">{html.escape(empty_message)}</p>'
    view = df.tail(max_rows).copy()
    return view.to_html(index=False, classes='report-table', border=0, escape=True, float_format=lambda x: f'{x:.6g}')


def _metric_card(label, value):
    return f'<div class="trqam-kpi"><div class="label">{html.escape(str(label))}</div><div class="value">{html.escape(str(value))}</div></div>'


def _last_value(df, key, default='n/a'):
    if df.empty or key not in df.columns:
        return default
    return _format_float(df.iloc[-1][key])


def build_report_html(cfg, save_dir, eval_df, offline_df, online_df, report_figures):
    final_success = _last_value(eval_df, 'success')
    final_return = _last_value(eval_df, 'episode.return')
    final_length = _last_value(eval_df, 'episode.length')
    final_flow_loss = _last_value(offline_df, 'actor/flow_loss')
    final_fast_loss = _last_value(offline_df, 'actor/fast_loss')
    final_path_kl = _last_value(offline_df, 'actor/path_kl')

    cards = ''.join([
        _metric_card('Run profile', RUN_PROFILE),
        _metric_card('Environment', cfg.env_name),
        _metric_card('Seed', cfg.seed),
        _metric_card('Offline steps', cfg.offline_steps),
        _metric_card('Online steps', cfg.online_steps),
        _metric_card('JAX backend', runtime_info.get('jax_backend', 'n/a')),
        _metric_card('Success', final_success),
        _metric_card('Return', final_return),
        _metric_card('Episode length', final_length),
        _metric_card('Flow loss', final_flow_loss),
        _metric_card('Fast loss', final_fast_loss),
        _metric_card('Path KL', final_path_kl),
    ])

    config_table = pd.DataFrame([(key, fmt_value(value)) for key, value in asdict(cfg).items()], columns=['Item', 'Value']).to_html(
        index=False,
        classes='report-table',
        border=0,
        escape=True,
    )

    figures_html = ''
    for title, rel_path in report_figures.items():
        figures_html += f'<h3>{html.escape(title)}</h3><img src="{html.escape(rel_path)}" alt="{html.escape(title)}" style="max-width:100%; border:1px solid #d0d7de; border-radius:6px;">'
    if not figures_html:
        figures_html = '<p class="trqam-muted">No figures were generated.</p>'

    return f'''
<div class="trqam-report">
  <h2>TRQAM Experiment Report</h2>
  <p class="trqam-muted">Generated at: {html.escape(time.strftime('%Y-%m-%d %H:%M:%S'))}</p>

  <div class="trqam-callout">
    This report was generated automatically from the current notebook run. For the full paper-style pipeline,
    run BC pretraining first, then run TRQAM fine-tuning from the saved BC checkpoint.
  </div>

  <h3>Key Metrics</h3>
  <div class="trqam-kpi-grid">{cards}</div>

  <h3>Method Summary</h3>
  <p>
    TRQAM starts from a flow policy and improves its vector field using an adjoint target derived from the Q-function action gradient.
    A path-space KL trust region and dual variable limit how far the fine-tuned policy can move from the pretrained base policy.
  </p>

  <h3>Experiment Configuration</h3>
  {config_table}

  <h3>Final Evaluation Log</h3>
  {_df_to_report_html(eval_df, 'No evaluation logs.', max_rows=1)}

  <h3>Recent Offline Training Logs</h3>
  {_df_to_report_html(offline_df, 'No offline training logs.', max_rows=5)}

  <h3>Recent Online Training Logs</h3>
  {_df_to_report_html(online_df, 'No online training logs.', max_rows=5)}

  <h3>Figures</h3>
  {figures_html}

  <h3>Interpretation Guide</h3>
  <p>
    `actor/flow_loss` tracks behavior-cloning quality, `actor/path_kl` and `dual/lambda` track trust-region usage,
    `critic/critic_loss` tracks Bellman backup stability, and `success` / `episode.return` summarize environment-level performance.
  </p>

  <p class="trqam-muted">Run directory: {html.escape(str(save_dir))}</p>
</div>
'''


def render_report_summary(cfg, save_dir, eval_df, offline_df, online_df):
    report_html = build_report_html(cfg, save_dir, eval_df, offline_df, online_df, report_figures)
    display(HTML(report_html))

    full_html = f'''<!doctype html>
<html lang="en">
<head>
<meta charset="utf-8">
<title>TRQAM Report</title>
{REPORT_CSS}
</head>
<body style="max-width: 980px; margin: 32px auto; padding: 0 20px;">
{report_html}
</body>
</html>
'''
    report_path = save_dir / 'report.html'
    report_path.write_text(full_html, encoding='utf-8')
    display(Markdown(f'Report HTML saved to: `{report_path}`'))
    return report_path


report_path = render_report_summary(cfg, save_dir, eval_df, offline_df, online_df)
